In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import random
import torch
import torchvision
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import PIL
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# 모델과 데이터셋을 GPU에 적재해야 빠른 연산이 가능합니다 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

root_path = '/kaggle/input/2026-rcv-winter-urp-vgg-demo'
classes = sorted(os.listdir('/kaggle/input/2026-rcv-winter-urp-vgg-demo/TRAIN'))


In [4]:
# 데이터 로드하는법 설명해야 할지도 
submit = pd.read_csv('/kaggle/input/2026-rcv-winter-urp-vgg-demo/sample-submission.csv') 
submit

root_path = '/kaggle/input/2026-rcv-winter-urp-vgg-demo'
submit = pd.read_csv(os.path.join(root_path, 'sample-submission.csv'))
train_dir = os.path.join(root_path, 'TRAIN')
classes = sorted(os.listdir(train_dir))
label_map = {v: k for k, v in enumerate(classes)}
label_map_rev = {v: k for k, v in label_map.items()}

In [5]:
class CustomDataset(Dataset):
    def __init__(self, rootpath, split, label_map, transform):
        self.rootpath = rootpath
        self.split = split.upper()
        self.label_map = label_map
        self.transform = transform
        self.image_list = []
        self.label_list = []

        if self.split == 'TRAIN':
            for cls in sorted(os.listdir(os.path.join(self.rootpath, self.split))):
                img_names = os.listdir(os.path.join(self.rootpath, self.split, cls))
                for img_name in img_names:
                    self.image_list.append(os.path.join(self.rootpath, self.split, cls, img_name))
                    self.label_list.append(cls)
        
        elif self.split == 'TEST':
            test_list = list(submit['name'])
            for name in test_list:
                self.image_list.append(os.path.join(self.rootpath, self.split, name))

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, index):
        img = PIL.Image.open(self.image_list[index]).convert('RGB')
        img = self.transform(img)
        if self.split == 'TRAIN':
            label = self.label_map[self.label_list[index]]
            return img, label
        return img

In [53]:

print(label_map,label_map_rev)

{'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5} {0: 'buildings', 1: 'forest', 2: 'glacier', 3: 'mountain', 4: 'sea', 5: 'street'}


In [6]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = CustomDataset(rootpath=root_path, split='TRAIN', label_map=label_map, transform=transform)
test_dataset = CustomDataset(rootpath=root_path, split='TEST', label_map=label_map, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False, num_workers=2)

In [7]:
# VGG16 모델 생성
class VGG16(nn.Module):
    def __init__(self, num_classes=6):
        super(VGG16, self).__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096), nn.ReLU(inplace=True), nn.Dropout(),
            nn.Linear(4096, 4096), nn.ReLU(inplace=True), nn.Dropout(),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model = VGG16(num_classes=6).to(device)

In [8]:
#사전 학습된 가중치 옮겨심기
pretrained_vgg = torchvision.models.vgg16(weights='IMAGENET1K_V1')
pretrained_dict = pretrained_vgg.state_dict()
model_dict = model.state_dict() # 라이브러리 models가 아닌 인스턴스 model 사용


new_dict = {}

for k, v in pretrained_dict.items():
    
    if k in model_dict and v.size() == model_dict[k].size():
        new_dict[k] = v

pretrained_dict = new_dict

# 내 모델의 가중치를 사전학습 가중치로 갱신
model_dict.update(pretrained_dict)
model.load_state_dict(model_dict)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 222MB/s] 


<All keys matched successfully>

In [11]:
print(pretrained_dict.keys())

dict_keys(['features.0.weight', 'features.0.bias', 'features.2.weight', 'features.2.bias', 'features.5.weight', 'features.5.bias', 'features.7.weight', 'features.7.bias', 'features.10.weight', 'features.10.bias', 'features.12.weight', 'features.12.bias', 'features.14.weight', 'features.14.bias', 'features.17.weight', 'features.17.bias', 'features.19.weight', 'features.19.bias', 'features.21.weight', 'features.21.bias', 'features.24.weight', 'features.24.bias', 'features.26.weight', 'features.26.bias', 'features.28.weight', 'features.28.bias', 'classifier.0.weight', 'classifier.0.bias', 'classifier.3.weight', 'classifier.3.bias'])


In [57]:
Epochs = 3
loss = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(),lr=0.001,momentum=0.9)


In [58]:
model.train() # 학습 모드 
for epoch in range(Epochs):
    epoch_loss=0
    epoch_acc=0
    for x, y in tqdm(train_loader):

        # 입력 데이터와 레이블은 cuda에서 연산 진행 
        x=torch.FloatTensor(x).to(device) 
        y=torch.LongTensor(y).to(device)
        
        out=model(x) # 예측값 
        cost=loss(out, y) # 손실값 
         
        optimizer.zero_grad() # gradient 초기화
        cost.backward()  # 역전파로 gradient 계산
        optimizer.step() # 가중치 업데이트 
        
        epoch_loss += cost.item()  
        epoch_acc += (torch.argmax(F.softmax(out, dim=1), dim=1) == y).sum()
        
    if epoch%1==0:
        print(f'{epoch+1} {epoch_loss} {epoch_acc/len(train_loader.dataset)*100}')

100%|██████████| 220/220 [01:48<00:00,  2.03it/s]


1 74.34915405511856 88.02906799316406


100%|██████████| 220/220 [01:48<00:00,  2.03it/s]


2 40.507580518722534 93.6083755493164


100%|██████████| 220/220 [01:48<00:00,  2.03it/s]

3 32.427315913140774 94.96223449707031


In [59]:
model.eval()
pred = []
with torch.no_grad():
    for x in tqdm(test_loader):
        x = x.to(device)
        out=model(x)
        pred_idx = torch.argmax(F.softmax(out, dim=1), dim=1) # 예측값은 클래스 ID별 확률값으로 저장되기 때문에, 가장 높은 확률값의 클래스 ID를 예측값으로 사용
        pred.extend(pred_idx)

100%|██████████| 47/47 [00:07<00:00,  6.11it/s]


In [60]:
final_pred = []

for i in range(len(pred)):
    predict = pred[i].detach().cpu().numpy()
    final_pred.append(label_map_rev[predict.item()]) # 클래스 ID -> 클래스 이름 변환 해줘야됩니다!
submit['label'] = final_pred
submit.to_csv('submit.csv', index=False)

In [61]:
print(submit)

           name      label
0     20057.jpg  buildings
1     20058.jpg   mountain
2     20059.jpg    glacier
3     20060.jpg  buildings
4     20061.jpg  buildings
...         ...        ...
2994  24325.jpg        sea
2995  24328.jpg   mountain
2996  24329.jpg   mountain
2997  24332.jpg     street
2998  24334.jpg    glacier

[2999 rows x 2 columns]
